# METADATA HAM1000

In [ ]:
df = pd.read_csv('/content/HAM10000_metadata.csv')
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

# PREPROCESSING

In [ ]:
# Complete labels
label_map = {
    'nv'   : 'Melanocytic Nevi',
    'mel'  : 'Melanoma',
    'bkl'  : 'Benign Keratosis',
    'bcc'  : 'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratosis',
    'vasc' : 'Vascular Lesion',
    'df'   : 'Dermatofibroma'
}
df['dx'] = df['dx'].map(label_map)

print(f"Total data : {len(df)} sampel")
print(f"column      : {df.columns.tolist()}")
df.head()

In [ ]:
print(f"Median age: {df['age'].median()}")

In [ ]:
median_age = df['age'].median()
df['age'].fillna(median_age, inplace=True)
print(f"Missing values after filling: {df['age'].isnull().sum()}")

In [ ]:
print('Variable distribution \'sex\':')
print(df['sex'].value_counts())

plt.figure(figsize=(8, 6))
sns.countplot(x='sex', data=df, palette='viridis')
plt.title('Gender Distribution')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

In [ ]:
# Identify indices of 'unknown' sex
unknown_indices = df[df['sex'] == 'unknown'].index

if len(unknown_indices) == 0:
    print("No 'unknown' values found in the 'sex' column.")
else:
    print(f"Number of 'unknown' samples before imputation: {len(unknown_indices)}")

    # Calculate overall male/female proportions for fallback
    known_sex_counts = df[df['sex'] != 'unknown']['sex'].value_counts(normalize=True)
    overall_male_prop = known_sex_counts.get('male', 0.5)
    overall_female_prop = known_sex_counts.get('female', 0.5)

    np.random.seed(42) # For reproducibility

    for idx in unknown_indices:
        current_dx = df.loc[idx, 'dx']
        current_loc = df.loc[idx, 'localization']

        # Filter known sexes for the current dx and localization group
        group_known_sexes = df[(df['dx'] == current_dx) &
                               (df['localization'] == current_loc) &
                               (df['sex'] != 'unknown')]['sex']

        if not group_known_sexes.empty:
            # Calculate proportions within the group
            group_sex_counts = group_known_sexes.value_counts(normalize=True)
            male_prop = group_sex_counts.get('male', 0)
            female_prop = group_sex_counts.get('female', 0)

            # If one is zero, adjust so proportions sum to 1
            if male_prop == 0 and female_prop == 0: # Should not happen if not empty, but for safety
                 assigned_sex = np.random.choice(['male', 'female'], p=[overall_male_prop, overall_female_prop])
            elif male_prop == 0:
                assigned_sex = 'female'
            elif female_prop == 0:
                assigned_sex = 'male'
            else:
                assigned_sex = np.random.choice(['male', 'female'], p=[male_prop, female_prop])
        else:
            # Fallback to overall proportions if no known sexes in this group
            assigned_sex = np.random.choice(['male', 'female'], p=[overall_male_prop, overall_female_prop])

        df.loc[idx, 'sex'] = assigned_sex

    print("Distribution of 'sex' variable after proportional imputation:")
    print(df['sex'].value_counts())

# IMAGE DATASET HAM1000

In [ ]:
extract_path = '/content/HAM10000'
os.makedirs(extract_path, exist_ok=True) # Ensure the directory exists

zip_file_part1 = '/content/HAM10000_images_part_1.zip'
zip_file_part2 = '/content/HAM10000_images_part_2.zip'

print("Extracting HAM10000_images_part_1.zip...")
with zipfile.ZipFile(zip_file_part1, 'r') as z:
    z.extractall(extract_path)
print("Part 1 extraction completed.")

print("\nExtracting HAM10000_images_part_2.zip...")
with zipfile.ZipFile(zip_file_part2, 'r') as z:
    z.extractall(extract_path)
print("Part 2 extraction completed.")

# Verify total images after both extractions
img_files = [f for f in os.listdir(extract_path) if f.endswith('.jpg')]
print(f"\nTotal images after extracting both parts: {len(img_files)}")

In [ ]:
extract_path = '/content/HAM10000'

print("Extracting parts 1 and 2...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extraction completed!")

# Verify total images
img_files = [f for f in os.listdir(extract_path) if f.endswith('.jpg')]
print(f"Total images: {len(img_files)}")

In [ ]:
# Check folder contents
all_files = os.listdir(extract_path)
print(f"Total files   : {len(all_files)}")
print(f"Sample files  : {all_files[:5]}")

# Count image files
img_files = [f for f in all_files if f.endswith('.jpg')]
print(f"Total images  : {len(img_files)}")

In [ ]:
# Filter out rows without image paths and sample a few images
sample_images = df.dropna(subset=['img_path']).sample(n=6, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (idx, row) in enumerate(sample_images.iterrows()):
    img_path = row['img_path']
    image_id = row['image_id']
    dx_label = row['dx']

    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f"ID: {image_id}\nDx: {dx_label}", fontsize=10)
    axes[i].axis('off')

plt.suptitle('Sample Images from Dataset', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# LABELING

In [ ]:
METADATA_PATH     = df
IMAGE_SOURCE_DIRS = ['/content/HAM10000']
OUTPUT_BASE_DIR   = '/content/drive/MyDrive/big data/HAM1000'

# Load metadata
def sanitize(label: str) -> str:
    return label.strip().replace(' ', '_').replace('/', '_')

# Create a folder for each label
for label in df['dx'].unique():
    os.makedirs(os.path.join(OUTPUT_BASE_DIR, sanitize(label)), exist_ok=True)

# Find image in source folders
def find_image(image_id: str) -> str | None:
    for src_dir in IMAGE_SOURCE_DIRS:
        path = os.path.join(src_dir, f'{image_id}.jpg')
        if os.path.exists(path):
            return path
    return None

# Copy images to Drive
success, failed = 0, 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Copying to Drive"):
    dest = os.path.join(OUTPUT_BASE_DIR, sanitize(row['dx']), f"{row['image_id']}.jpg")
    if os.path.exists(dest):
        success += 1
        continue
    src = find_image(row['image_id'])
    if src:
        try:
            shutil.copy2(src, dest)
            success += 1
        except Exception:
            failed += 1
    else:
        failed += 1

print(f"Completed — success: {success} | failed: {failed}")

In [ ]:
image_data_dir = '/content/drive/MyDrive/big data/HAM1000'

if os.path.exists(image_data_dir):
    print(f"Directory '{image_data_dir}' found.")

    subdirectories = [d for d in os.listdir(image_data_dir) if os.path.isdir(os.path.join(image_data_dir, d))]
    print(f"Total of {len(subdirectories)} sub-directories (dx labels) found:")

    total_images_in_organized_dir = 0
    for subdir in sorted(subdirectories):
        subdir_path = os.path.join(image_data_dir, subdir)
        images_in_subdir = [f for f in os.listdir(subdir_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        print(f"  - '{subdir}': {len(images_in_subdir)} images")
        total_images_in_organized_dir += len(images_in_subdir)

    print(f"\nTotal of all images in '{image_data_dir}': {total_images_in_organized_dir}")
else:
    print(f"Directory '{image_data_dir}' not found. Please ensure the data has been copied correctly.")

# METADATA DERM12345

In [ ]:
df1 = pd.read_csv('/content/derm12345_metadata_test.csv')
df1.head()

In [ ]:
df1.shape

In [ ]:
df2 = pd.read_csv('/content/derm12345_metadata_train.csv')
df2.head()

In [ ]:
df2.shape

# PREPROCESSING

In [ ]:
merged = pd.concat([df1, df2], ignore_index=True)
print("Shape of merged dataset:", merged.shape)
print("Head of merged dataset:")
display(merged.head())
print("Tail of merged dataset:")
display(merged.tail())

In [ ]:
merged.info()

In [ ]:
merged.isnull().sum()

In [ ]:
merged.duplicated().sum()

In [ ]:
print(f"Duplicate image_id  : {merged['image_id'].duplicated().sum()}")
print(f"Duplicate patient_id: {merged['patient_id'].duplicated().sum()}")

In [ ]:
merged = merged.drop('split', axis=1)
merged.info()

# IMAGE DATASET

In [ ]:
extract_path = '/content/dermoscopic'
os.makedirs(extract_path, exist_ok=True) # Ensure the directory exists

zip_file_train_part1 = '/content/derm12345_train_part_1.zip'
zip_file_train_part2 = '/content/derm12345_train_part_2.zip'
zip_file_test = '/content/derm12345_test.zip'

print("Extracting derm12345_train_part_1.zip...")
with zipfile.ZipFile(zip_file_train_part1, 'r') as z:
    z.extractall(extract_path)
print("Train part 1 extraction completed.")

print("\nExtracting derm12345_train_part_2.zip...")
with zipfile.ZipFile(zip_file_train_part2, 'r') as z:
    z.extractall(extract_path)
print("Train part 2 extraction completed.")

print("\nExtracting derm12345_test.zip...")
with zipfile.ZipFile(zip_file_test, 'r') as z:
    z.extractall(extract_path)
print("Test extraction completed.")

In [ ]:
folder_counts = {}
for folder in sorted(os.listdir(extract_path)):
    folder_path = os.path.join(extract_path, folder)
    if os.path.isdir(folder_path):
        img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        folder_counts[folder] = len(img_files)

print(f"Total of {len(folder_counts)} sub-directories found:")
for folder, count in folder_counts.items():
    print(f"  - '{folder}': {count} images")

In [ ]:
# Ensure all image_ids have corresponding image files
image_files = set()
for folder in os.listdir(extract_path):
    folder_path = os.path.join(extract_path, folder)
    if os.path.isdir(folder_path):
        for f in os.listdir(folder_path):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_files.add(os.path.splitext(f)[0])  # without extension

df_ids = set(merged['image_id'])
missing = df_ids - image_files
orphan = image_files - df_ids

print(f"image_id in CSV but missing image file  : {len(missing)}")
print(f"Image file exists but missing in CSV    : {len(orphan)}")